In [2]:
import joblib
model = joblib.load("../models/heart_lr.pkl")
print(type(model))
print(type(model).__name__)

<class 'sklearn.calibration.CalibratedClassifierCV'>
CalibratedClassifierCV


# 07 — SHAP Explainability
## MediSight AI: A Transparency Label for Healthcare Prediction Systems

---

### What is this notebook?

This notebook generates **SHAP (SHapley Additive exPlanations)** values
for all 4 disease prediction models in MediSight AI.

SHAP tells us:
- **Which features** had the most impact on a prediction
- **Which direction** they pushed the prediction (towards disease or away)
- **How much** each feature contributed to the final score

---

### What SHAP is NOT

SHAP is not a model. It is an **explanation technique** that works on top
of already-trained models. Our models are already saved in `../models/`.
This notebook just reads them and explains them.

---

### What this notebook produces

For each disease (Diabetes, Heart, Liver, Kidney), we generate:

| Output File | What it is |
|---|---|
| `{disease}_shap.json` | Feature importance numbers — for the dashboard |
| `{disease}_shap_beeswarm.png` | Plot showing all features and their SHAP values |
| `{disease}_shap_bar.png` | Bar chart of top features by average importance |
| `{disease}_shap_waterfall.png` | Explanation for one specific patient |
| `{disease}_shap_force.html` | Interactive explanation (optional) |
| `{disease}_explainer.pkl` | Saved explainer for live dashboard use |

---

### Before running this notebook

Make sure:
1. scikit-learn version is **1.8.0** (run `pip install scikit-learn==1.8.0`)
2. You are using **Python 3.11** kernel in Jupyter
3. All model `.pkl` files exist in `../models/`
4. All `.npy` data files exist in `../datasets/processed/`

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
c:\Users\neman\AppData\Local\Programs\Python\Python311\python.exe


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS AND SETUP
# All libraries needed for SHAP explainability
# ─────────────────────────────────────────────────────────────────────────────

import shap                      # main SHAP library
import numpy as np               # for array operations
import pandas as pd              # for DataFrames (SHAP needs feature names)
import joblib                    # to load .pkl model files
import json                      # to save SHAP results as JSON
import matplotlib                # plotting backend
import matplotlib.pyplot as plt  # for saving plots
import os                        # for creating folders
import warnings

# Suppress non-critical warnings during SHAP computation
warnings.filterwarnings("ignore")

# Use non-interactive backend so plots save to file instead of popping up
matplotlib.use("Agg")

# Create the explainability output folder if it doesn't exist
os.makedirs("../explainability", exist_ok=True)

print("All imports successful")
print(f"   SHAP version: {shap.__version__}")
print(f"   Output folder: ../explainability/")

All imports successful
   SHAP version: 0.51.0
   Output folder: ../explainability/


---

## Step 1 — Configuration

Before we run anything, we define which model to use for each disease.

We have 3 models per disease (LR, RF, XGB). We pick the one with the
best AUC-ROC from `all_metrics.json`:

| Disease | Best Model | Why |
|---|---|---|
| Diabetes | XGBoost | Highest AUC (0.828) |
| Heart Disease | Logistic Regression | Best recall (78%) — catches most cases |
| Liver Disease | Random Forest | Highest AUC (0.937) |
| Kidney Disease | Logistic Regression | All models get 100% — LR is simplest |

### Two types of SHAP explainers

SHAP has different explainers for different model types:

- **TreeExplainer** → for XGBoost and Random Forest (tree-based models)
- **LinearExplainer** → for Logistic Regression (linear model)

You cannot use TreeExplainer on a linear model, or vice versa.
The config below tracks which explainer type to use per disease.

### What is CalibratedClassifierCV?

Some of our models were wrapped in `CalibratedClassifierCV` during
training (to improve probability estimates). For LinearExplainer,
we need to unwrap this wrapper to get to the actual LogisticRegression
inside. The code handles this automatically.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# Defines which model to use for SHAP for each disease
# ─────────────────────────────────────────────────────────────────────────────

DISEASE_CONFIG = {

    "diabetes": {
        "model_key":    "xgb",       # use XGBoost model file: diabetes_xgb.pkl
        "model_type":   "tree",      # → will use TreeExplainer
        "display_name": "Diabetes"
    },

    "heart": {
        "model_key":    "lr",        # use LR model file: heart_lr.pkl
        "model_type":   "linear",    # → will use LinearExplainer
        "display_name": "Heart Disease",
        "calibrated":   True         # this model is wrapped in CalibratedClassifierCV
    },

    "liver": {
        "model_key":    "rf",        # use Random Forest: liver_rf.pkl
        "model_type":   "tree",      # → will use TreeExplainer
        "display_name": "Liver Disease"
    },

    "kidney": {
        "model_key":    "lr",        # use LR model file: kidney_lr.pkl
        "model_type":   "linear",    # → will use LinearExplainer
        "display_name": "Chronic Kidney Disease",
        "calibrated":   True         # this model is also wrapped
    }
}

# Print summary
print("Disease configuration:")
print(f"{'Disease':<12} {'Model':<8} {'Explainer Type'}")
print("-" * 40)
for disease, config in DISEASE_CONFIG.items():
    print(f"{disease:<12} {config['model_key'].upper():<8} "
          f"{config['model_type'].capitalize()}Explainer")

Disease configuration:
Disease      Model    Explainer Type
----------------------------------------
diabetes     XGB      TreeExplainer
heart        LR       LinearExplainer
liver        RF       TreeExplainer
kidney       LR       LinearExplainer


---

## Step 2 — Helper Functions

These are utility functions used by the main pipeline.
You do not call these directly — the pipeline calls them automatically.

### Function overview

| Function | What it does |
|---|---|
| `load_disease_data()` | Loads model, scaler, features, and test data |
| `unwrap_calibrated_model()` | Extracts LR from inside CalibratedClassifierCV |
| `create_explainer()` | Creates the correct SHAP explainer for the model type |
| `compute_shap_values()` | Runs SHAP and returns a clean 2D array |

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────


def load_disease_data(disease_key):
    """
    Loads everything needed for SHAP computation for one disease.

    Inputs (all loaded from disk):
        - The trained model (.pkl)
        - The fitted scaler (.pkl)
        - The list of feature names (.pkl)
        - The test set X and y (.npy arrays)

    Returns:
        model, scaler, features (list), X_test (array), y_test (array)
    """
    model_key = DISEASE_CONFIG[disease_key]["model_key"]

    print(f"  Loading model:    ../models/{disease_key}_{model_key}.pkl")
    model = joblib.load(f"../models/{disease_key}_{model_key}.pkl")

    print(f"  Loading scaler:   ../models/{disease_key}_scaler.pkl")
    scaler = joblib.load(f"../models/{disease_key}_scaler.pkl")

    print(f"  Loading features: ../datasets/processed/"
          f"{disease_key}_features.pkl")
    features = joblib.load(
        f"../datasets/processed/{disease_key}_features.pkl"
    )

    print(f"  Loading test data: ../datasets/processed/"
          f"{disease_key}_X_test.npy")
    X_test = np.load(f"../datasets/processed/{disease_key}_X_test.npy")
    y_test = np.load(f"../datasets/processed/{disease_key}_y_test.npy")

    print(f" Loaded: {len(features)} features, "
          f"X_test shape = {X_test.shape}")
    return model, scaler, features, X_test, y_test


def unwrap_calibrated_model(model):
    """
    Extracts the base LogisticRegression from inside a
    CalibratedClassifierCV wrapper.

    Why needed:
        LinearExplainer requires a raw sklearn model, not a wrapper.
        CalibratedClassifierCV wraps the model to improve probabilities,
        but SHAP cannot look inside it directly.

    How it works:
        CalibratedClassifierCV stores multiple calibrated versions
        internally. We take the first one and extract its .estimator.

    Returns:
        The unwrapped base model (LogisticRegression)
    """
    if hasattr(model, "calibrated_classifiers_"):
        base = model.calibrated_classifiers_[0].estimator
        print(f"  Unwrapped CalibratedClassifierCV "
              f"→ {type(base).__name__}")
        return base
    else:
        # Model is not wrapped — use directly
        print(f"  Model is not calibrated, using directly: "
              f"{type(model).__name__}")
        return model


def create_explainer(model, model_type, X_test, disease_key):
    """
    Creates the correct SHAP explainer for the model type.

    TreeExplainer  → XGBoost, Random Forest
                     Fast, exact SHAP values
                     Works directly on tree models

    LinearExplainer → Logistic Regression
                      Needs a background dataset (sample of X_test)
                      Requires the raw base model (not CalibratedClassifierCV)

    Returns:
        shap.TreeExplainer or shap.LinearExplainer
    """
    config = DISEASE_CONFIG[disease_key]

    if model_type == "tree":
        print("  Creating TreeExplainer (for XGB/RF)...")
        explainer = shap.TreeExplainer(model)

    elif model_type == "linear":
        # Step 1: unwrap if calibrated
        if config.get("calibrated", False):
            base_model = unwrap_calibrated_model(model)
        else:
            base_model = model

        # Step 2: create background dataset
        # LinearExplainer needs a reference distribution
        # We use a random sample of 100 test points (or fewer if small dataset)
        n_background = min(100, X_test.shape[0])
        background = shap.sample(X_test, n_background)
        print(f"  Background dataset: {n_background} samples")

        print("  Creating LinearExplainer (for LR)...")
        explainer = shap.LinearExplainer(base_model, background)

    else:
        raise ValueError(f"Unknown model_type: '{model_type}'. "
                         f"Use 'tree' or 'linear'.")

    print(f" Explainer ready: {type(explainer).__name__}")
    return explainer


def compute_shap_values(explainer, X_test, model_type, disease_key):
    """
    Computes SHAP values for the test set.

    Why we limit samples for TreeExplainer:
        TreeExplainer is slower on large datasets.
        500 samples is enough to compute reliable feature importance.
        LinearExplainer is faster so we use all samples.

    Why the output shape needs handling:
        For binary classification, SHAP sometimes returns:
        - A list of 2 arrays [class_0_values, class_1_values]
        - A single 3D array of shape (n_samples, n_features, n_classes)
        - A single 2D array of shape (n_samples, n_features)

        We always want class 1 (disease = positive), so we extract
        the right slice depending on what shape we get.

    Returns:
        shap_values: 2D array of shape (n_samples, n_features)
        X_sample:    the input data used (same n_samples as shap_values)
    """
    # Limit samples for tree models (speed optimisation)
    if model_type == "tree":
        n_samples = min(500, X_test.shape[0])
        X_sample  = X_test[:n_samples]
        print(f"  Using {n_samples} samples (tree explainer — "
              f"speed optimised)")
    else:
        X_sample = X_test
        print(f"  Using all {X_test.shape[0]} samples "
              f"(linear explainer — fast)")

    print("  Computing SHAP values... (may take a few minutes)")
    raw = explainer.shap_values(X_sample)

    # Handle different output formats from SHAP
    if isinstance(raw, list):
        # Got a list → binary classification returns [class_0, class_1]
        # We want class 1 (disease present)
        shap_vals = raw[1] if len(raw) > 1 else raw[0]
        print(f"  Output was list of {len(raw)} arrays → using class 1")

    elif isinstance(raw, np.ndarray) and raw.ndim == 3:
        # Got a 3D array of shape (samples, features, classes)
        # We want class 1
        shap_vals = raw[:, :, 1]
        print(f"  Output was 3D array → slicing class 1")

    else:
        # Already a 2D array — use directly
        shap_vals = raw
        print(f"  Output was 2D array → using directly")

    print(f" SHAP values shape: {shap_vals.shape}")
    return shap_vals, X_sample


print("All helper functions defined")

All helper functions defined


---

## Step 3 — Output Functions

These functions take the computed SHAP values and save them to disk.

### Why we save to JSON

The dashboard (`dashboard/app.py`) needs to display feature importance
without recomputing SHAP every time. The JSON gives Karan a simple
file to read:

```json
{
    "top_5_features": ["BMI", "GenHlth", "Age", ...],
    "feature_importance": {
        "BMI": 0.1234,
        "GenHlth": 0.0987,
        ...
    }
}
```

### Why we save the explainer as .pkl

For **live predictions** in the dashboard (Tab 1 — Transparency Label
and Tab 4 — What-If Simulator), Karan needs to compute SHAP values
for a single new patient entered by the user. The saved explainer
allows this without retraining or recomputing from scratch.

### The three plots explained

| Plot | What it shows | Used for |
|---|---|---|
| Beeswarm | Every sample as a dot, coloured by feature value | Presentation |
| Bar chart | Average importance across all samples | Dashboard Tab 3 |
| Waterfall | One patient — step by step how features added/removed risk | Dashboard Tab 1 & 5 |

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT FUNCTIONS
# Each function saves one type of output to ../explainability/
# ─────────────────────────────────────────────────────────────────────────────


def save_shap_json(disease_key, model_key, features,
                   shap_values, explainer):
    """
    Saves SHAP feature importance as a JSON file.

    This is the primary output that the dashboard reads.
    It contains:
    - Feature names sorted by importance (most important first)
    - The importance value (mean absolute SHAP) for each feature
    - Top 5 and Top 10 feature lists for quick access
    - The base value (what the model predicts with no features)

    Output: ../explainability/{disease_key}_shap.json
    """

    # Mean absolute SHAP value per feature
    # abs() because direction doesn't matter for overall importance
    # mean() because we average across all test samples
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    # Build a dict: {feature_name: importance_value}
    importance_dict = {
        feat: round(float(val), 6)
        for feat, val in zip(features, mean_abs_shap)
    }

    # Sort by importance (highest first)
    importance_sorted = dict(
        sorted(importance_dict.items(),
               key=lambda x: x[1], reverse=True)
    )

    # Get the base value (model output when all features = 0)
    # This is used for waterfall and force plots
    if hasattr(explainer, "expected_value"):
        ev = explainer.expected_value
        if isinstance(ev, (list, np.ndarray)):
            base_value = float(ev[1]) if len(ev) > 1 else float(ev[0])
        else:
            base_value = float(ev)
    else:
        base_value = 0.0

    # Build the full JSON output
    output = {
        "disease":            disease_key,
        "model":              model_key,
        "base_value":         base_value,
        "feature_importance": importance_sorted,
        "top_5_features":     list(importance_sorted.keys())[:5],
        "top_10_features":    list(importance_sorted.keys())[:10],
        "n_features":         len(features),
        "n_samples_used":     int(shap_values.shape[0])
    }

    path = f"../explainability/{disease_key}_shap.json"
    with open(path, "w") as f:
        json.dump(output, f, indent=4)

    print(f"  Saved JSON: {path}")
    print(f"     Top 5 features: {output['top_5_features']}")
    return output


def save_beeswarm_plot(disease_key, features, shap_values, X_sample):
    """
    Saves a SHAP beeswarm plot.

    What it shows:
        Each dot is one patient from the test set.
        The x-axis shows the SHAP value (how much this feature
        pushed the prediction left or right).
        The colour shows the actual feature value
        (red = high value, blue = low value).

    How to read it:
        A red dot on the right means: high feature value increases risk.
        A blue dot on the right means: low feature value increases risk.

    Output: ../explainability/{disease_key}_shap_beeswarm.png
    """
    X_df = pd.DataFrame(X_sample, columns=features)

    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values,
        X_df,
        plot_type="dot",    # "dot" = beeswarm
        max_display=15,     # show top 15 features
        show=False          # do not display in notebook — save to file
    )
    plt.title(
        f"{DISEASE_CONFIG[disease_key]['display_name']} — "
        f"SHAP Feature Impact (Beeswarm)",
        fontsize=14, fontweight="bold", pad=15
    )
    plt.tight_layout()

    path = f"../explainability/{disease_key}_shap_beeswarm.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved beeswarm plot: {path}")


def save_bar_plot(disease_key, features, shap_values, X_sample):
    """
    Saves a SHAP bar chart (global feature importance).

    What it shows:
        Each bar is one feature.
        Bar length = mean absolute SHAP value across all patients.
        Longer bar = more important feature overall.

    This is the clearest chart for presentations and reports.
    It answers: "Which features matter most for this disease?"

    Output: ../explainability/{disease_key}_shap_bar.png
    """
    X_df = pd.DataFrame(X_sample, columns=features)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values,
        X_df,
        plot_type="bar",    # "bar" = horizontal bar chart
        max_display=15,
        show=False
    )
    plt.title(
        f"{DISEASE_CONFIG[disease_key]['display_name']} — "
        f"Global Feature Importance (SHAP)",
        fontsize=14, fontweight="bold", pad=15
    )
    plt.tight_layout()

    path = f"../explainability/{disease_key}_shap_bar.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved bar chart: {path}")


def save_waterfall_plot(disease_key, features,
                        shap_values, X_sample,
                        explainer, patient_idx=0):
    """
    Saves a SHAP waterfall plot for one specific patient.

    What it shows:
        Starts at the base value (what the model predicts on average).
        Each feature either adds to or subtracts from that starting point.
        Ends at the final prediction probability for this patient.

    Example:
        Base value: 0.35 (35% risk on average)
        + BMI = 32:     +0.12 (pushes risk up)
        + Age = 62:     +0.08 (pushes risk up)
        - Exercise = 1: -0.05 (lowers risk)
        Final: 0.50 (50% risk for this patient)

    patient_idx=0 means we explain the first patient in the test set.
    You can change this to any index to explain a different patient.

    Output: ../explainability/{disease_key}_shap_waterfall.png
    """
    # Get the base value from the explainer
    if hasattr(explainer, "expected_value"):
        ev = explainer.expected_value
        if isinstance(ev, (list, np.ndarray)):
            base_val = float(ev[1]) if len(ev) > 1 else float(ev[0])
        else:
            base_val = float(ev)
    else:
        base_val = 0.0

    X_df = pd.DataFrame(X_sample, columns=features)

    # Create a SHAP Explanation object for one patient
    # This bundles together: SHAP values + base value + actual feature values
    shap_exp = shap.Explanation(
        values=shap_values[patient_idx],        # SHAP values for this patient
        base_values=base_val,                   # starting point
        data=X_df.iloc[patient_idx].values,     # actual feature values
        feature_names=features                  # feature labels for the plot
    )

    plt.figure(figsize=(10, 8))
    shap.waterfall_plot(
        shap_exp,
        max_display=12,  # show top 12 contributing features
        show=False
    )
    plt.title(
        f"{DISEASE_CONFIG[disease_key]['display_name']} — "
        f"Single Patient Explanation (Test Patient #{patient_idx})",
        fontsize=13, fontweight="bold", pad=15
    )
    plt.tight_layout()

    path = f"../explainability/{disease_key}_shap_waterfall.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved waterfall plot: {path}")


def save_force_plot_html(disease_key, features,
                         shap_values, X_sample,
                         explainer, patient_idx=0):
    """
    Saves an interactive HTML force plot for one patient.

    What it shows:
        Same information as waterfall but interactive in a browser.
        Features pushing risk up appear on the right (red).
        Features pushing risk down appear on the left (blue).

    This is optional — used for the dashboard if Karan wants
    to embed an interactive explanation.

    Output: ../explainability/{disease_key}_shap_force.html
    """
    try:
        if hasattr(explainer, "expected_value"):
            ev = explainer.expected_value
            if isinstance(ev, (list, np.ndarray)):
                base_val = float(ev[1]) if len(ev) > 1 else float(ev[0])
            else:
                base_val = float(ev)
        else:
            base_val = 0.0

        shap.initjs()
        force = shap.force_plot(
            base_val,
            shap_values[patient_idx],
            pd.DataFrame(X_sample, columns=features).iloc[patient_idx],
            show=False
        )
        path = f"../explainability/{disease_key}_shap_force.html"
        shap.save_html(path, force)
        print(f"  Saved force plot HTML: {path}")

    except Exception as e:
        # Force plot is optional — if it fails, skip it
        print(f"  Force plot skipped (non-critical): {e}")


def save_explainer_object(disease_key, explainer):
    """
    Saves the SHAP explainer object to disk as a .pkl file.

    Why this is important:
        When a user enters patient data in the dashboard and clicks
        'Generate Transparency Label', Karan's code needs to compute
        SHAP values for that specific patient in real time.

        Instead of retraining the explainer every time, Karan loads
        this saved explainer and calls:
            shap_vals = explainer.shap_values(new_patient_data)

    Output: ../explainability/{disease_key}_explainer.pkl
    """
    path = f"../explainability/{disease_key}_explainer.pkl"
    joblib.dump(explainer, path)
    print(f"  Saved explainer object: {path}")


print("All output functions defined")

All output functions defined


---

## Step 4 — Master Pipeline

This is the function that runs everything for one disease.

It calls all the helper functions and output functions in order:
1. Load data (model, scaler, features, test set)
2. Create SHAP explainer (TreeExplainer or LinearExplainer)
3. Compute SHAP values
4. Save JSON
5. Save beeswarm plot
6. Save bar chart
7. Save waterfall plot
8. Save force plot HTML (optional)
9. Save explainer .pkl

After defining this function, we call it once per disease in the
cells below.

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# MASTER PIPELINE
# Runs the full SHAP workflow for one disease
# ─────────────────────────────────────────────────────────────────────────────

def run_shap_pipeline(disease_key):
    """
    Runs the complete SHAP explainability pipeline for one disease.

    Steps:
        1. Load model, scaler, features, test data
        2. Create appropriate SHAP explainer
        3. Compute SHAP values
        4. Save JSON summary
        5. Save beeswarm plot
        6. Save bar chart
        7. Save waterfall plot
        8. Save force plot HTML (optional)
        9. Save explainer .pkl for dashboard use

    Args:
        disease_key: one of "diabetes", "heart", "liver", "kidney"

    Returns:
        A dict containing all computed objects (for verification use)
    """
    config     = DISEASE_CONFIG[disease_key]
    model_key  = config["model_key"]
    model_type = config["model_type"]

    print(f"\n{'='*60}")
    print(f"PROCESSING: {config['display_name'].upper()}")
    print(f"Model: {model_key.upper()} | "
          f"Explainer: {model_type.capitalize()}Explainer")
    print(f"{'='*60}")

    # ── 1. Load all data ──────────────────────────────────────────────
    print("\n[1/9] Loading data...")
    model, scaler, features, X_test, y_test = \
        load_disease_data(disease_key)

    # ── 2. Create SHAP explainer ──────────────────────────────────────
    print("\n[2/9] Creating SHAP explainer...")
    explainer = create_explainer(
        model, model_type, X_test, disease_key
    )

    # ── 3. Compute SHAP values ────────────────────────────────────────
    print("\n[3/9] Computing SHAP values...")
    shap_values, X_sample = compute_shap_values(
        explainer, X_test, model_type, disease_key
    )

    # ── 4. Save JSON ──────────────────────────────────────────────────
    print("\n[4/9] Saving SHAP JSON...")
    shap_summary = save_shap_json(
        disease_key, model_key, features,
        shap_values, explainer
    )

    # ── 5. Save beeswarm plot ─────────────────────────────────────────
    print("\n[5/9] Saving beeswarm plot...")
    save_beeswarm_plot(
        disease_key, features, shap_values, X_sample
    )

    # ── 6. Save bar chart ─────────────────────────────────────────────
    print("\n[6/9] Saving bar chart...")
    save_bar_plot(
        disease_key, features, shap_values, X_sample
    )

    # ── 7. Save waterfall plot ────────────────────────────────────────
    print("\n[7/9] Saving waterfall plot...")
    save_waterfall_plot(
        disease_key, features, shap_values,
        X_sample, explainer, patient_idx=0
    )

    # ── 8. Save force plot HTML (optional) ────────────────────────────
    print("\n[8/9] Saving force plot HTML...")
    save_force_plot_html(
        disease_key, features, shap_values,
        X_sample, explainer, patient_idx=0
    )

    # ── 9. Save explainer for dashboard ──────────────────────────────
    print("\n[9/9] Saving explainer object for dashboard...")
    save_explainer_object(disease_key, explainer)

    # ── Done ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"{config['display_name'].upper()} COMPLETE")
    print(f"   Top 5 features: {shap_summary['top_5_features']}")
    print(f"{'='*60}")

    return {
        "disease":     disease_key,
        "model":       model_key,
        "shap_values": shap_values,
        "X_sample":    X_sample,
        "features":    features,
        "explainer":   explainer,
        "summary":     shap_summary
    }


print("Pipeline function defined — ready to run")

Pipeline function defined — ready to run


---

## Step 5 — Run the Pipeline

We now run the pipeline for each disease one at a time.

**Run each cell separately.** Do not run all at once.
Wait for each to finish before moving to the next.

### Expected runtime
| Disease | Model | Approx Time |
|---|---|---|
| Diabetes | XGBoost (500 samples) | 2–4 minutes |
| Heart | Logistic Regression (all samples) | 1–2 minutes |
| Liver | Random Forest (500 samples) | 3–5 minutes |
| Kidney | Logistic Regression (80 samples) | ~30 seconds |

If a cell takes more than 10 minutes, something is wrong.
Stop it and check the error.

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN DIABETES
# Model: XGBoost | Explainer: TreeExplainer
# Expected time: 2–4 minutes
# ─────────────────────────────────────────────────────────────────────────────

result_diabetes = run_shap_pipeline("diabetes")


PROCESSING: DIABETES
Model: XGB | Explainer: TreeExplainer

[1/9] Loading data...
  Loading model:    ../models/diabetes_xgb.pkl
  Loading scaler:   ../models/diabetes_scaler.pkl


FileNotFoundError: [Errno 2] No such file or directory: '../models/diabetes_scaler.pkl'

In [12]:
import os

print("=== models/ folder ===")
for f in sorted(os.listdir("../models")):
    print(f"  {f}")

print("\n=== datasets/processed/ folder ===")
for f in sorted(os.listdir("../datasets/processed")):
    print(f"  {f}")

=== models/ folder ===
  .gitkeep.txt
  all_metrics.json
  diabetes_lr.pkl
  diabetes_rf.pkl
  diabetes_xgb.pkl
  heart_lr.pkl
  heart_rf.pkl
  heart_rf_threshold.pkl
  heart_scaler.pkl
  heart_xgb.pkl
  kidney_lr.pkl
  kidney_rf.pkl
  kidney_scaler.pkl
  kidney_xgb.pkl
  liver_lr.pkl
  liver_rf.pkl
  liver_scaler.pkl
  liver_xgb.pkl

=== datasets/processed/ folder ===
  diabetes_X_test.npy
  diabetes_X_train.npy
  diabetes_clean.csv
  diabetes_features.pkl
  diabetes_y_test.npy
  diabetes_y_train.npy
  heart_X_test.npy
  heart_X_train.npy
  heart_clean.csv
  heart_features.pkl
  heart_y_test.npy
  heart_y_train.npy
  kidney_X_test.npy
  kidney_X_train.npy
  kidney_clean.csv
  kidney_features.pkl
  kidney_y_test.npy
  kidney_y_train.npy
  liver_X_test.npy
  liver_X_train.npy
  liver_clean.csv
  liver_features.pkl
  liver_y_test.npy
  liver_y_train.npy


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# PATCH — Rebuild and save missing diabetes_scaler.pkl
# Run this ONCE, then never again
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib

# Load the training data that Prajwal already saved
X_train = np.load("../datasets/processed/diabetes_X_train.npy")

# Fit the scaler on training data (exactly what Prajwal should have done)
scaler = StandardScaler()
scaler.fit(X_train)

# Save it where the pipeline expects it
joblib.dump(scaler, "../models/diabetes_scaler.pkl")

print("diabetes_scaler.pkl saved to ../models/")
print(f"   Fitted on X_train shape: {X_train.shape}")

diabetes_scaler.pkl saved to ../models/
   Fitted on X_train shape: (56554, 21)
